## Mr.HelpMate AI: Insurance Policy RAG System

### Part 1: Introduction and Setup

Project Objective: Build a robust generative search system capable of answering questions accurately from the "Group Member Life Insurance Policy" document.

### Architecture:

- Embedding Layer: Ingesting the PDF, chunking the text, and storing it in a ChromaDB vector database.

- Search Layer: Implementing semantic search, a caching mechanism for efficiency, and a Cross-Encoder re-ranker to surface the most relevant context.

- Generation Layer: Using an LLM with heavily engineered prompts and few-shot examples to generate accurate, context-bound answers.


## 1.1 Installing and Importing Required Libraries

Necessary packages for PDF processing, vector databases, embeddings, and re-ranking.

In [1]:
# Install required libraries
%pip install -U -q pypdf chromadb sentence-transformers cross-encoder openai tenacity requests python-dotenv

ERROR: Could not find a version that satisfies the requirement cross-encoder (from versions: none)
ERROR: No matching distribution found for cross-encoder
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## Using Gemini API Key instead of OpenAPI, however with openAI's sdk compatibility

In [2]:

import os
import requests
import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI
from tenacity import retry, wait_random_exponential, stop_after_attempt
from dotenv import load_dotenv

# Load environment variables from a .env file
load_dotenv()

# Fetch the Gemini API key
my_api_key = os.getenv("GEMINI_API_KEY")

# Initialize the OpenAI client pointing to the Gemini endpoint
gemini_client = OpenAI(
    api_key=my_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

print("Libraries imported and Gemini Client initialized successfully!")


/Users/jags/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Libraries imported and Gemini Client initialized successfully!


Stage 1: The Embedding Layer
In this layer, we will download the policy document, clean the text, split it into overlapping chunks (to preserve context), and embed them using a HuggingFace SentenceTransformer model to save on API costs and ensure fast local embedding.

1.2 Download and Process the PDF

In [4]:
# Download the PDF
pdf_url = "https://cdn.upgrad.com/uploads/production/585ca56a-6fe1-4b93-903c-1c1a1de74bf1/Principal-Sample-Life-Insurance-Policy.pdf"
pdf_filename = "Life_Insurance_Policy.pdf"

response = requests.get(pdf_url)
with open(pdf_filename, 'wb') as f:
    f.write(response.content)
print("PDF downloaded successfully.")

# Extract text from PDF
def extract_text_from_pdf(filepath):
    reader = PdfReader(filepath)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    return text

policy_text = extract_text_from_pdf(pdf_filename)
print(f"Extracted {len(policy_text)} characters from the document.")

PDF downloaded successfully.
Extracted 105357 characters from the document.


1.3 Chunking and Vector Database Initialization

We will chunk the text using a sliding window approach. Overlapping chunks ensure that sentences at the edge of a chunk don't lose their meaning.

In [5]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

chunks = chunk_text(policy_text, chunk_size=150, overlap=30)
print(f"Split document into {len(chunks)} chunks.")

# Initialize Embedding Model and ChromaDB
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

client = chromadb.Client()
# Clear collection if it already exists to avoid duplicates during testing
try:
    client.delete_collection(name="insurance_policy")
except:
    pass

collection = client.create_collection(name="insurance_policy")

# Generate embeddings and store in ChromaDB
embeddings = embedding_model.encode(chunks).tolist()

collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)
print("Chunks embedded and loaded into ChromaDB successfully.")

Split document into 142 chunks.
Chunks embedded and loaded into ChromaDB successfully.


Stage 2: The Search Layer
This layer handles retrieving the most relevant information.

Cache Check: We use a simple dictionary to check if the query was asked before.

Semantic Search: We retrieve the top 10 chunks from ChromaDB.

Re-ranking: We use a CrossEncoder to re-rank those 10 chunks and pick the absolute best 3 to send to the LLM.

2.1 Implementing Search, Cache, and Re-ranking

In [6]:
# Initialize Re-ranker
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Initialize a simple cache dictionary
query_cache = {}

def search_and_rerank(query, top_k_retrieve=10, top_k_rerank=3):
    """
    Retrieves chunks from ChromaDB, re-ranks them using a cross-encoder,
    and utilizes a simple semantic cache.
    """
    # 1. Check Cache
    if query in query_cache:
        print("-> Result served from Cache!")
        return query_cache[query]

    # 2. Embed Query and Retrieve top 10 from ChromaDB
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k_retrieve
    )
    retrieved_chunks = results['documents'][0]

    # 3. Re-rank using Cross-Encoder
    # Create pairs of (query, chunk) for the cross-encoder
    pairs = [[query, chunk] for chunk in retrieved_chunks]
    scores = cross_encoder.predict(pairs)

    # Sort chunks by cross-encoder scores in descending order
    scored_chunks = list(zip(retrieved_chunks, scores))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)

    # Extract the top 3 chunks
    best_chunks = [chunk for chunk, score in scored_chunks[:top_k_rerank]]

    # Update Cache
    query_cache[query] = best_chunks

    return best_chunks

Stage 3: The Generation Layer
Inspired by the get_chat_completions() function from your training notebook, we will use OpenAI's ChatCompletion API wrapped in a Tenacity retry block. The system prompt is heavily engineered to restrict hallucinations.

3.1 Defining the LLM Call and Prompts

In [14]:
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def get_chat_completions(messages):
    # Use the 'client' object we initialized earlier
    response = gemini_client.chat.completions.create(
        model='gemini-2.0-flash', # Updated to a Gemini model
        messages=messages,
        temperature=0.0, # 0.0 ensures highly deterministic and factual outputs
    )
    return response.choices[0].message.content

def generate_answer(query, retrieved_context):
    """
    Constructs the prompt with context and few-shot examples, then calls the LLM.
    """
    context_string = "\n\n".join([f"Chunk {i+1}:\n{chunk}" for i, chunk in enumerate(retrieved_context)])

    system_prompt = f"""
    You are an expert Life Insurance Agent named Mr.HelpMate AI.
    Your goal is to answer user questions based STRICTLY on the provided policy document context.

    Guidelines:
    1. If the answer is not contained in the provided context, gracefully state: "I am sorry, but the provided policy document does not contain the answer to this question."
    2. Do not use outside knowledge or hallucinate information.
    3. Keep your answers clear, professional, and concise.
    4. When applicable, cite the specific conditions mentioned in the text.

    ---
    FEW-SHOT EXAMPLES:
    User: How many days do I have to cancel the policy?
    Context: The policyholder has a "free look" period of 10 days from the date of receipt to review and cancel the policy for a full refund.
    Assistant: According to the policy, you have a "free look" period of 10 days from the date of receipt to cancel the policy and receive a full refund.
    ---

    CONTEXT FROM DOCUMENT:
    {context_string}
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]

    return get_chat_completions(messages)

Part 2: Evaluation and Outputs (For Screenshots)
Below, we define the 3 self-designed queries. The code will loop through them, print the Top 3 Retrieved Chunks (for your first set of screenshots), and print the Final LLM Generated Answer (for your second set of screenshots).

Suggested Queries based on standard life insurance policies:

What are the eligibility requirements for a Member to be insured?

What happens if the premium is not paid by the end of the grace period?

What is the definition of "Proof of Good Health"?

2.1 Execution Cell

In [15]:
queries = [
    "What are the criteria for a Member to be eligible for insurance?",
    "What happens if I fail to pay the premium before the grace period expires?",
    "What constitutes Proof of Good Health according to this policy?"
]

for i, query in enumerate(queries, 1):
    print("="*80)
    print(f"QUERY {i}: {query}")
    print("="*80)


    top_chunks = search_and_rerank(query)

    print("\n" + "*"*30 + " TOP 3 RETRIEVED CHUNKS " + "*"*30)
    for j, chunk in enumerate(top_chunks, 1):
        print(f"\n--- Rank {j} ---")
        print(chunk.strip())

    final_answer = generate_answer(query, top_chunks)

    print("\n" + "*"*30 + " FINAL GENERATED ANSWER" + "*"*30)
    print(final_answer)
    print("\n\n")

QUERY 1: What are the criteria for a Member to be eligible for insurance?
-> Result served from Cache!

****************************** TOP 3 RETRIEVED CHUNKS ******************************

--- Rank 1 ---
applicable premium rates in effect on the Policy Anniversary. This policy has been updated effective January 1, 2014 PART III - INDIVIDUAL REQUIREMENTS AND RIGHTS GC 6006 Section A - Eligibility, Page 1 PART III - INDIVIDUAL REQUIREMENTS AND RIGHTS Section A - Eligibility Article 1 - Member Life Insurance A person will be eligible for Member Life Insurance on the date the person completes 30 consecutive days of continuous Active Work with the Policyholder as a Member. In no circumstance will a person be eligible for Member Life Insurance under this Group Policy if the person is eligible under any other Group Term Life Insurance policy underwritten by The Principal. Article 2 - Member Accidental Death and Dismemberment Insurance A person will be eligible for Member Accidental Death and

2.2 Test the Cache

Run this cell to prove that the caching mechanism works instantly without re-embedding or querying ChromaDB.

In [16]:
print("Testing Cache Execution...")
cache_test_query = queries[0]
print(f"Query: {cache_test_query}")
cached_chunks = search_and_rerank(cache_test_query)
# Output should explicitly print "-> Result served from Cache!"

Testing Cache Execution...
Query: What are the criteria for a Member to be eligible for insurance?
-> Result served from Cache!
